# SAM 3D Objects on SageMaker — click an object, get a 3D model

Reproduces the open-source backend of Meta's [`convert-image-to-3d`](https://www.aidemos.meta.com/segment-anything/editor/convert-image-to-3d) demo: **point at an object in a photo → it gets masked → a full 3D model (a 3D Gaussian Splat) is generated**, with a turntable preview.

Deploys [SAM 3D Objects](https://github.com/facebookresearch/sam-3d-objects) behind a SageMaker **async** endpoint and drives it end to end.

### How to use this notebook (non-expert friendly)
Run the cells **top to bottom, once**. Sections 2–6 are one-time setup (IAM, quota, weights, image, deploy); sections 7–10 are the actual 3D generation you can repeat. Each setup cell either does the work for you or prints exactly what to hand an admin if your role lacks permission.

### Why a custom container?
SAM 3D Objects needs a CUDA-toolkit-compiled stack (`pytorch3d`, `flash_attn`, `kaolin`, `gsplat`, `MoGe`) that can't ride a stock PyTorch DLC, so we use a **Bring-Your-Own-Container** image (`container/`). It compiles on AWS CodeBuild (no Docker needed on this notebook). Weights (~13 GB, gated) are staged to S3 once and mounted at `/opt/ml/model`.

### Instance & cost
`ml.g6e.2xlarge` (1× NVIDIA L40S, 48 GB). SAM 3D Objects requires **≥ 32 GB VRAM**, which rules out the 24 GB A10G/L4 cards; L40S/`g6e` is AWS's most cost-efficient GenAI GPU and clears the floor on a single GPU (~$2.4/hr on-demand). **Delete the endpoint when idle** (section 11).

## 0. Client-side prerequisites
These libraries run in the notebook (not on the endpoint).

In [ ]:
%pip install -q 'sagemaker>=2.240' 'boto3>=1.34' sagemaker-studio-image-build pillow numpy matplotlib imageio
%pip install -q 'transformers>=4.40' torch   # local click->mask SAM (CPU is fine)
print('deps installed (restart the kernel if pip asks you to).')

## 1. Configuration

In [ ]:
import boto3, sagemaker, json

sess = sagemaker.Session()
region = sess.boto_region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
role = sagemaker.get_execution_role()              # the Studio/notebook execution role
bucket = sess.default_bucket()                     # sagemaker-<region>-<account>

ECR_REPO       = 'sam3d-objects'
IMAGE_URI      = f'{account_id}.dkr.ecr.{region}.amazonaws.com/{ECR_REPO}:latest'
WEIGHTS_PREFIX = f's3://{bucket}/sam3d/weights/'
INPUT_PREFIX   = f's3://{bucket}/sam3d-inputs/'
OUTPUT_PREFIX  = f's3://{bucket}/sam3d-outputs/'
FAILURE_PREFIX = f's3://{bucket}/sam3d-failures/'
ENDPOINT_NAME  = 'sam3d-objects-g6e'
INSTANCE_TYPE  = 'ml.g6e.2xlarge'                  # 1x L40S 48GB; >=32GB VRAM required

role_name = role.split(':role/')[-1].split('/')[-1]  # IAM RoleName (no path)
print('region   :', region)
print('account  :', account_id)
print('role     :', role, '\n  -> RoleName:', role_name)
print('image    :', IMAGE_URI)
print('endpoint :', ENDPOINT_NAME, '/', INSTANCE_TYPE)

## 2. One-time IAM setup (run me first)
The build (CodeBuild → ECR) and the deploy (endpoint pulls the image + passes the role) need specific permissions on **your execution role**. This cell:
1. adds `codebuild.amazonaws.com` to the role's **trust policy** (keeping SageMaker), and
2. attaches an inline policy `Sam3dBuildAndDeploy` granting CodeBuild, ECR push/pull, S3, Logs, and `PassRole`.

If your role isn't allowed to modify itself (common), the cell prints the role name, the exact JSON, and CLI commands to give an admin. Nothing else runs until this is in place.

In [ ]:
iam = boto3.client('iam')

trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': ['sagemaker.amazonaws.com', 'codebuild.amazonaws.com']},
        'Action': 'sts:AssumeRole',
    }],
}

# Covers BOTH the sm-docker/CodeBuild build and the g6e endpoint image pull.
perm_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {'Sid': 'CodeBuild', 'Effect': 'Allow',
         'Action': ['codebuild:CreateProject', 'codebuild:DeleteProject',
                    'codebuild:StartBuild', 'codebuild:BatchGetBuilds'],
         'Resource': 'arn:aws:codebuild:*:*:project/sagemaker-studio*'},
        {'Sid': 'CodeBuildLogs', 'Effect': 'Allow',
         'Action': ['logs:CreateLogGroup', 'logs:CreateLogStream',
                    'logs:GetLogEvents', 'logs:PutLogEvents'],
         'Resource': '*'},
        {'Sid': 'EcrPushPull', 'Effect': 'Allow',
         'Action': ['ecr:CreateRepository', 'ecr:DescribeRepositories', 'ecr:DescribeImages',
                    'ecr:ListImages', 'ecr:BatchCheckLayerAvailability', 'ecr:GetDownloadUrlForLayer',
                    'ecr:BatchGetImage', 'ecr:InitiateLayerUpload', 'ecr:UploadLayerPart',
                    'ecr:CompleteLayerUpload', 'ecr:PutImage'],
         'Resource': ['arn:aws:ecr:*:*:repository/sagemaker-studio*',
                      f'arn:aws:ecr:*:*:repository/{ECR_REPO}*']},
        {'Sid': 'EcrAuth', 'Effect': 'Allow', 'Action': 'ecr:GetAuthorizationToken', 'Resource': '*'},
        {'Sid': 'S3Context', 'Effect': 'Allow',
         'Action': ['s3:GetObject', 's3:PutObject', 's3:DeleteObject', 's3:ListBucket'],
         'Resource': ['arn:aws:s3:::sagemaker-*', 'arn:aws:s3:::sagemaker-*/*']},
        {'Sid': 'S3CreateBucket', 'Effect': 'Allow', 'Action': 's3:CreateBucket',
         'Resource': 'arn:aws:s3:::sagemaker*'},
        {'Sid': 'IamRead', 'Effect': 'Allow', 'Action': ['iam:GetRole', 'iam:ListRoles'], 'Resource': '*'},
        {'Sid': 'PassRole', 'Effect': 'Allow', 'Action': 'iam:PassRole',
         'Resource': 'arn:aws:iam::*:role/*',
         'Condition': {'StringLikeIfExists': {
             'iam:PassedToService': ['codebuild.amazonaws.com', 'sagemaker.amazonaws.com']}}},
    ],
}

def _cli_fallback(err):
    print('\nCould NOT modify the role automatically:\n  ', err)
    print('\nGive these to someone with IAM admin (or run with admin creds):\n')
    with open('sam3d_trust.json', 'w') as f: json.dump(trust_policy, f, indent=2)
    with open('sam3d_policy.json', 'w') as f: json.dump(perm_policy, f, indent=2)
    print(f'aws iam update-assume-role-policy --role-name {role_name} \\\n'
          f'    --policy-document file://sam3d_trust.json')
    print(f'aws iam put-role-policy --role-name {role_name} --policy-name Sam3dBuildAndDeploy \\\n'
          f'    --policy-document file://sam3d_policy.json')
    print('\n(JSON written to sam3d_trust.json and sam3d_policy.json in this folder.)')

try:
    iam.update_assume_role_policy(RoleName=role_name, PolicyDocument=json.dumps(trust_policy))
    iam.put_role_policy(RoleName=role_name, PolicyName='Sam3dBuildAndDeploy',
                        PolicyDocument=json.dumps(perm_policy))
    print('IAM ready: trust + Sam3dBuildAndDeploy attached to', role_name)
    print('(IAM changes can take ~10-20s to propagate.)')
except Exception as e:
    _cli_fallback(e)

## 3. Check the `ml.g6e.2xlarge` endpoint quota
New accounts often have a **0** quota for g6e endpoint usage, which makes the deploy in section 6 fail. This checks it and, if it's 0, gives you the one-line request.

In [ ]:
QUOTA_NAME = 'ml.g6e.2xlarge for endpoint usage'
try:
    sq = boto3.client('service-quotas', region_name=region)
    found = None
    for page in sq.get_paginator('list_service_quotas').paginate(ServiceCode='sagemaker'):
        for q in page['Quotas']:
            if q['QuotaName'] == QUOTA_NAME:
                found = q; break
        if found: break
    if not found:
        print('Quota not found via API; check the console:')
        print(f'  https://{region}.console.aws.amazon.com/servicequotas/home/services/sagemaker/quotas')
    else:
        val = found['Value']
        print(f"{QUOTA_NAME}: {val}  (code {found['QuotaCode']})")
        if val < 1:
            print('\n>>> Quota is 0 — request an increase to at least 1, then wait for approval:')
            print(f"    aws service-quotas request-service-quota-increase --service-code sagemaker \\\n"
                  f"        --quota-code {found['QuotaCode']} --desired-value 1 --region {region}")
        else:
            print('Quota is sufficient to deploy one endpoint instance.')
except Exception as e:
    print('Could not read Service Quotas (servicequotas:ListServiceQuotas may be missing).')
    print('Check manually:', f'https://{region}.console.aws.amazon.com/servicequotas/home/services/sagemaker/quotas')
    print('detail:', e)

## 4. Stage the gated checkpoints to S3 (one-time)
`facebook/sam-3d-objects` is **gated**: accept the license at https://huggingface.co/facebook/sam-3d-objects , then use an access-granted token (`HF_TOKEN`). ~13 GB. A terminal is faster/resumable, but you can run it here.

In [ ]:
import os
# Set your access-granted token, then uncomment the staging lines.
# os.environ['HF_TOKEN'] = 'hf_...'
# assert os.environ.get('HF_TOKEN'), 'set HF_TOKEN (access-granted) first'
# !pip install -q 'huggingface_hub[cli]>=0.26,<1.0'
# !python prepare_weights.py --out checkpoints
# !aws s3 sync checkpoints/ {WEIGHTS_PREFIX}
!aws s3 ls {WEIGHTS_PREFIX} --recursive --human-readable --summarize 2>/dev/null | tail -n 20 \
    || echo 'No weights staged yet — uncomment the lines above and run.'

## 5. Build & push the image with CodeBuild (one-time)
No Docker on this notebook, so we use the SageMaker Studio Image Build CLI, which runs the build on CodeBuild and pushes to ECR. The cell below calls it **in-process** (so it works even when `sm-docker` isn't on the shell PATH) and changes into `container/` for you. The image compiles pytorch3d / flash_attn / gsplat from source — **a clean build takes ~20 min**; `BUILD_GENERAL1_2XLARGE` gives it the RAM it needs.

Re-running this cell overwrites `sam3d-objects:latest` in ECR with a fresh, correct image (e.g. after any change under `container/serve/`). Then redeploy in section 6.

> **Watch it from a terminal** (this build is long): `python monitor.py build --follow`

### 5a. Write the correct container files (run before building)
This regenerates `container/Dockerfile`, `container/serve/predictor.py`, `container/serve/sam3d_handler.py`, and the launcher from known-good content embedded in this notebook, and deletes the stale `inference.py`. It makes the build self-contained — you do **not** need to separately upload the `container/` folder.

In [ ]:
# Write the CORRECT container files into ./container before building, so the build
# never depends on which files happen to be uploaded to this environment. This
# permanently fixes the 'module inference has no attribute reconstruct' error:
# the handler is sam3d_handler.py (unique name), and predictor imports it.
import base64, os, shutil, json

_FILES = json.loads(r"""{"Dockerfile": "IyBTQU0gM0QgT2JqZWN0cyDigJQgU2FnZU1ha2VyIEJyaW5nLVlvdXItT3duLUNvbnRhaW5lciBpbWFnZS4KIwojIFJlcHJvZHVjZXMgaHR0cHM6Ly9naXRodWIuY29tL2ZhY2Vib29rcmVzZWFyY2gvc2FtLTNkLW9iamVjdHMvYmxvYi9tYWluL2RvYy9zZXR1cC5tZAojIChjb25kYSBlbnYgKyBDVURBLXRvb2xraXQtY29tcGlsZWQgcHl0b3JjaDNkIC8gZmxhc2hfYXR0biAvIGthb2xpbiAvIGdzcGxhdCksCiMgdGhlbiBsYXllcnMgYSB0aW55IEZsYXNrIGFwcCB0aGF0IHNhdGlzZmllcyB0aGUgU2FnZU1ha2VyIC9waW5nICsgL2ludm9jYXRpb25zCiMgY29udHJhY3QuIFdlaWdodHMgYXJlIE5PVCBiYWtlZCBpbiDigJQgdGhleSBhcmUgbW91bnRlZCBmcm9tIGFuIFMzIHByZWZpeCBhdAojIC9vcHQvbWwvbW9kZWwgYXQgZGVwbG95IHRpbWUuCiMKIyBCdWlsZCBmb3IgbGludXgvYW1kNjQgKFNhZ2VNYWtlciBpcyB4ODYtNjQpLiBUaGUgY29tcGlsZSBzdGVwcyBhcmUgc2xvdyBhbmQKIyBSQU0taHVuZ3J5OyBzZWUgY29udGFpbmVyL2J1aWxkX2FuZF9wdXNoLnNoLgoKRlJPTSBjb25kYWZvcmdlL21pbmlmb3JnZTM6MjQuOS4yLTAKCkFSRyBTQU0zRF9DT01NSVQ9bWFpbgojIEw0MFMgKGc2ZSkgaXMgQWRhID0gc21fODkuIFJlc3RyaWN0IGFyY2ggdG8ga2VlcCB0aGUgZmxhc2hfYXR0bi9weXRvcmNoM2QvZ3NwbGF0CiMgYnVpbGRzIHRyYWN0YWJsZS4gQWRkIG1vcmUgKGUuZy4gIjguMDs4LjY7OC45OzkuMCIpIG9ubHkgaWYgeW91IHRhcmdldCBvdGhlciBHUFVzLgpBUkcgVE9SQ0hfQ1VEQV9BUkNIX0xJU1Q9IjguOSIKRU5WIFRPUkNIX0NVREFfQVJDSF9MSVNUPSR7VE9SQ0hfQ1VEQV9BUkNIX0xJU1R9CkVOViBERUJJQU5fRlJPTlRFTkQ9bm9uaW50ZXJhY3RpdmUKIyBCb3VuZCBzb3VyY2UtYnVpbGQgcGFyYWxsZWxpc20gc28gZmxhc2hfYXR0biBkb2Vzbid0IE9PTSB0aGUgYnVpbGRlci4KRU5WIE1BWF9KT0JTPTQKRU5WIFBJUF9OT19DQUNIRV9ESVI9MQoKIyBTeXN0ZW0gbGlicyBuZWVkZWQgYXQgcnVudGltZSBieSBvcGVuY3YgLyBvcGVuM2QgLyBicHkgLyByZW5kZXJpbmcuClJVTiBhcHQtZ2V0IHVwZGF0ZSAmJiBhcHQtZ2V0IGluc3RhbGwgLXkgLS1uby1pbnN0YWxsLXJlY29tbWVuZHMgXAogICAgICAgIGdpdCB3Z2V0IGNhLWNlcnRpZmljYXRlcyBidWlsZC1lc3NlbnRpYWwgXAogICAgICAgIGxpYmdsMSBsaWJnbGliMi4wLTAgbGlieHJlbmRlcjEgbGlieGV4dDYgbGlic202IFwKICAgICAgICBsaWJlZ2wxIGxpYmdsZXMyIGZmbXBlZyBcCiAgICAmJiBybSAtcmYgL3Zhci9saWIvYXB0L2xpc3RzLyoKCldPUktESVIgL29wdApSVU4gZ2l0IGNsb25lIGh0dHBzOi8vZ2l0aHViLmNvbS9mYWNlYm9va3Jlc2VhcmNoL3NhbS0zZC1vYmplY3RzLmdpdCBcCiAgICAmJiBjZCBzYW0tM2Qtb2JqZWN0cyBcCiAgICAmJiBnaXQgY2hlY2tvdXQgJHtTQU0zRF9DT01NSVR9CgpXT1JLRElSIC9vcHQvc2FtLTNkLW9iamVjdHMKCiMgMSkgQ3JlYXRlIHRoZSBjb25kYSBlbnYgZXhhY3RseSBhcyB1cHN0cmVhbSBzcGVjaWZpZXMgKENVREEgdG9vbGtpdCwgZ2NjLCBweXRob24gMy4xMSkuClJVTiBtYW1iYSBlbnYgY3JlYXRlIC1mIGVudmlyb25tZW50cy9kZWZhdWx0LnltbCAmJiBtYW1iYSBjbGVhbiAtYWZ5CgojIEFsbCBzdWJzZXF1ZW50IFJVTnMgZXhlY3V0ZSBpbnNpZGUgdGhlIHNhbTNkLW9iamVjdHMgZW52LgpTSEVMTCBbIm1hbWJhIiwgInJ1biIsICItbiIsICJzYW0zZC1vYmplY3RzIiwgIi9iaW4vYmFzaCIsICItYyJdCgojIDIpIENvcmUgKyBweXRvcmNoM2QgKHR3by1zdGVwLCBwZXIgdXBzdHJlYW06IHAzZCdzIHRvcmNoIHBpbiBpcyBicm9rZW4gb3RoZXJ3aXNlKS4KRU5WIFBJUF9FWFRSQV9JTkRFWF9VUkw9Imh0dHBzOi8vcHlwaS5uZ2MubnZpZGlhLmNvbSBodHRwczovL2Rvd25sb2FkLnB5dG9yY2gub3JnL3dobC9jdTEyMSIKUlVOIHBpcCBpbnN0YWxsIC1lICcuW2Rldl0nClJVTiBwaXAgaW5zdGFsbCAtZSAnLltwM2RdJwoKIyAzKSBJbmZlcmVuY2UgZXh0cmFzOiBrYW9saW4gKE5WSURJQSB3aGVlbHMgcGlubmVkIHRvIHRvcmNoIDIuNS4xL2N1MTIxKSwgZ3NwbGF0LCBldGMuCkVOViBQSVBfRklORF9MSU5LUz0iaHR0cHM6Ly9udmlkaWEta2FvbGluLnMzLnVzLWVhc3QtMi5hbWF6b25hd3MuY29tL3RvcmNoLTIuNS4xX2N1MTIxLmh0bWwiClJVTiBwaXAgaW5zdGFsbCAtZSAnLltpbmZlcmVuY2VdJwoKIyA0KSBQYXRjaCBoeWRyYSAodXBzdHJlYW0gZml4IG5vdCB5ZXQgcmVsZWFzZWQgdG8gUHlQSSkuClJVTiAuL3BhdGNoaW5nL2h5ZHJhCgojIDUpIFNlcnZpbmcgZGVwcyAoa2VwdCBtaW5pbWFsOyB0aGUgaGVhdnkgc3RhY2sgaXMgYWxyZWFkeSBpbiB0aGUgZW52KS4KUlVOIHBpcCBpbnN0YWxsICJmbGFzaz49My4wLDw0IiAiZ3VuaWNvcm4+PTIyLjAiICJib3RvMz49MS4zNCIKCiMgNikgT3VyIHNlcnZpbmcgY29kZS4gYG5vdGVib29rL2AgaG9sZHMgdGhlIHJlcG8ncyBpbmZlcmVuY2UucHkgKyB0aGUgc2FtM2Rfb2JqZWN0cwojICAgIGltcG9ydCBwYXRoOyB3ZSBhZGQgaXQgdG8gUFlUSE9OUEFUSCBzbyBwcmVkaWN0b3IucHkgY2FuIGltcG9ydCB0aGUgcGlwZWxpbmUuCkVOViBQWVRIT05QQVRIPSIvb3B0L3NhbS0zZC1vYmplY3RzOi9vcHQvc2FtLTNkLW9iamVjdHMvbm90ZWJvb2s6JHtQWVRIT05QQVRIfSIKRU5WIFNBTTNEX1JFUE89L29wdC9zYW0tM2Qtb2JqZWN0cwpDT1BZIHNlcnZlLyAvb3B0L3Byb2dyYW0vClJVTiBjaG1vZCAreCAvb3B0L3Byb2dyYW0vc2VydmUKCiMgU2FnZU1ha2VyIGNhbGxzIGBkb2NrZXIgcnVuIDxpbWFnZT4gc2VydmVgLiBNYWtlIC9vcHQvcHJvZ3JhbS9zZXJ2ZSByZXNvbHZhYmxlCiMgYW5kIGVuc3VyZSBpdCBhY3RpdmF0ZXMgdGhlIGNvbmRhIGVudiBiZWZvcmUgbGF1bmNoaW5nIGd1bmljb3JuLgpFTlYgUEFUSD0iL29wdC9wcm9ncmFtOiR7UEFUSH0iCkVOViBTQUdFTUFLRVJfUFJPR1JBTT1wcmVkaWN0b3IucHkKV09SS0RJUiAvb3B0L3Byb2dyYW0KCkVOVFJZUE9JTlQgWyIvb3B0L3Byb2dyYW0vc2VydmUiXQo=", ".dockerignore": "KiovX19weWNhY2hlX18vCioqLyoucHljCioqLyoucHlvCi5EU19TdG9yZQo=", "serve/predictor.py": "IiIiRmxhc2sgYXBwIGltcGxlbWVudGluZyB0aGUgU2FnZU1ha2VyIEJZT0Mgc2VydmluZyBjb250cmFjdC4KClNhZ2VNYWtlciBob3N0aW5nIHJlcXVpcmVzIHR3byByb3V0ZXMgb24gcG9ydCA4MDgwOgogIC0gR0VUICAvcGluZyAgICAgICAgIC0+IDIwMCB3aGVuIHRoZSBjb250YWluZXIgaXMgaGVhbHRoeQogIC0gUE9TVCAvaW52b2NhdGlvbnMgIC0+IHJ1biBpbmZlcmVuY2Ugb24gdGhlIHJlcXVlc3QgYm9keQoKRm9yIGFzeW5jIGVuZHBvaW50cyB0aGUgcmVxdWVzdCBib2R5IGlzIHRoZSBKU09OIHdlIFBVVCB0byBTMywgYW5kIHdoYXRldmVyIHdlCnJldHVybiBpcyB3cml0dGVuIHRvIHRoZSBTMyBPdXRwdXRMb2NhdGlvbi4gV2UgbG9hZCB0aGUgKGhlYXZ5KSBwaXBlbGluZSBsYXppbHkKb24gdGhlIGZpcnN0IHJlcXVlc3Qgc28gL3BpbmcgcmV0dXJucyAyMDAgcXVpY2tseSBkdXJpbmcgdGhlIFNhZ2VNYWtlcgpzdGFydHVwIGhlYWx0aCBjaGVjaywgYXZvaWRpbmcgYSBmYWxzZSAidW5oZWFsdGh5IiBkdXJpbmcgdGhlIGxvbmcgbW9kZWwgbG9hZC4KIiIiCgppbXBvcnQganNvbgppbXBvcnQgbG9nZ2luZwppbXBvcnQgdHJhY2ViYWNrCgppbXBvcnQgZmxhc2sKCmltcG9ydCBzYW0zZF9oYW5kbGVyIGFzIHNhbTNkICAjIE5PVCBgaW5mZXJlbmNlYCDigJQgdGhhdCBuYW1lIGNvbGxpZGVzIHdpdGggdGhlIHJlcG8ncyBub3RlYm9vay9pbmZlcmVuY2UucHkKCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJzYW0zZC5wcmVkaWN0b3IiKQpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyBbJShsZXZlbG5hbWUpc10gJShtZXNzYWdlKXMiKQoKYXBwID0gZmxhc2suRmxhc2soX19uYW1lX18pCgoKQGFwcC5yb3V0ZSgiL3BpbmciLCBtZXRob2RzPVsiR0VUIl0pCmRlZiBwaW5nKCk6CiAgICAjIEhlYWx0aHkgYXMgc29vbiBhcyB0aGUgcHJvY2VzcyBpcyB1cC4gVGhlIG1vZGVsIGxvYWRzIG9uIGZpcnN0IC9pbnZvY2F0aW9uczsKICAgICMgdGhlIFNhZ2VNYWtlciBhc3luYyBpbnZvY2F0aW9uIHRpbWVvdXQgKG5vdCB0aGUgaGVhbHRoIGNoZWNrKSBjb3ZlcnMgdGhhdC4KICAgIHJldHVybiBmbGFzay5SZXNwb25zZShyZXNwb25zZT0iXG4iLCBzdGF0dXM9MjAwLCBtaW1ldHlwZT0iYXBwbGljYXRpb24vanNvbiIpCgoKQGFwcC5yb3V0ZSgiL2ludm9jYXRpb25zIiwgbWV0aG9kcz1bIlBPU1QiXSkKZGVmIGludm9jYXRpb25zKCk6CiAgICBpZiBmbGFzay5yZXF1ZXN0LmNvbnRlbnRfdHlwZSBhbmQgImpzb24iIG5vdCBpbiBmbGFzay5yZXF1ZXN0LmNvbnRlbnRfdHlwZToKICAgICAgICByZXR1cm4gX2Vycm9yKDQxNSwgZiJVbnN1cHBvcnRlZCBjb250ZW50IHR5cGU6IHtmbGFzay5yZXF1ZXN0LmNvbnRlbnRfdHlwZX0iKQoKICAgIHRyeToKICAgICAgICBib2R5ID0gZmxhc2sucmVxdWVzdC5nZXRfZGF0YShhc190ZXh0PVRydWUpCiAgICAgICAgZGF0YSA9IGpzb24ubG9hZHMoYm9keSkgaWYgYm9keSBlbHNlIHt9CiAgICBleGNlcHQganNvbi5KU09ORGVjb2RlRXJyb3IgYXMgZXhjOgogICAgICAgIHJldHVybiBfZXJyb3IoNDAwLCBmIkludmFsaWQgSlNPTiBib2R5OiB7ZXhjfSIpCgogICAgdHJ5OgogICAgICAgIHJlc3VsdCA9IHNhbTNkLnJlY29uc3RydWN0KGRhdGEpCiAgICBleGNlcHQgVmFsdWVFcnJvciBhcyBleGM6ICAjIGNsaWVudCBlcnJvciAoYmFkL21pc3NpbmcgZmllbGRzKQogICAgICAgIHJldHVybiBfZXJyb3IoNDAwLCBzdHIoZXhjKSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBzZXJ2ZXIgLyBtb2RlbCBlcnJvcgogICAgICAgIGxvZ2dlci5lcnJvcigiSW5mZXJlbmNlIGZhaWxlZDogJXNcbiVzIiwgZXhjLCB0cmFjZWJhY2suZm9ybWF0X2V4YygpKQogICAgICAgIHJldHVybiBfZXJyb3IoNTAwLCBmIkluZmVyZW5jZSBmYWlsZWQ6IHtleGN9IikKCiAgICByZXR1cm4gZmxhc2suUmVzcG9uc2UoCiAgICAgICAgcmVzcG9uc2U9anNvbi5kdW1wcyhyZXN1bHQpLCBzdGF0dXM9MjAwLCBtaW1ldHlwZT0iYXBwbGljYXRpb24vanNvbiIKICAgICkKCgpkZWYgX2Vycm9yKGNvZGU6IGludCwgbWVzc2FnZTogc3RyKToKICAgIHJldHVybiBmbGFzay5SZXNwb25zZSgKICAgICAgICByZXNwb25zZT1qc29uLmR1bXBzKHsiZXJyb3IiOiBtZXNzYWdlfSksIHN0YXR1cz1jb2RlLCBtaW1ldHlwZT0iYXBwbGljYXRpb24vanNvbiIKICAgICkK", "serve/sam3d_handler.py": "IiIiU0FNIDNEIE9iamVjdHMgcmVjb25zdHJ1Y3Rpb24gbG9naWMgZm9yIHRoZSBTYWdlTWFrZXIgZW5kcG9pbnQuCgpOT1RFIG9uIHRoZSBtb2R1bGUgbmFtZTogdGhpcyBmaWxlIGlzIGludGVudGlvbmFsbHkgTk9UIGNhbGxlZCBgaW5mZXJlbmNlLnB5YC4KVGhlIHVwc3RyZWFtIHJlcG8gc2hpcHMgYG5vdGVib29rL2luZmVyZW5jZS5weWAgb24gUFlUSE9OUEFUSCwgc28gYSBoYW5kbGVyIG5hbWVkCmBpbmZlcmVuY2VgIGNvbGxpZGVzIHdpdGggaXQg4oCUIGBpbXBvcnQgaW5mZXJlbmNlYCB3b3VsZCByZXNvbHZlIHRvIHRoZSByZXBvJ3MKbW9kdWxlICh3aGljaCBoYXMgbm8gYHJlY29uc3RydWN0YCkgaW5zdGVhZCBvZiBvdXJzLiBLZWVwaW5nIGEgdW5pcXVlIG5hbWUgbGV0cwpgZnJvbSBpbmZlcmVuY2UgaW1wb3J0IC4uLmAgYmVsb3cgdW5hbWJpZ3VvdXNseSByZWFjaCB0aGUgcmVwbydzIHBpcGVsaW5lLgoKVGhpbiB3cmFwcGVyIG92ZXIgdGhlIHVwc3RyZWFtIHJlcG8ncyBgbm90ZWJvb2svaW5mZXJlbmNlLnB5YC4gR2l2ZW4gYW4gUkdCIGltYWdlCmFuZCBhIGJpbmFyeSBtYXNrIG1hcmtpbmcgT05FIG9iamVjdCwgaXQ6CgogIDEuIGxpZnRzIHRoZSBtYXNrZWQgb2JqZWN0IHRvIGEgM0QgR2F1c3NpYW4gU3BsYXQgIChJbmZlcmVuY2UuX19jYWxsX18pCiAgMi4gc2VyaWFsaXplcyBpdCB0byBhIC5wbHkKICAzLiBvcHRpb25hbGx5IHJlbmRlcnMgYSB0dXJudGFibGUgR0lGIHByZXZpZXcgICAgIChtYWtlX3NjZW5lICsgcmVuZGVyX3ZpZGVvKQoKVGhpcyBtaXJyb3JzIG5vdGVib29rL2RlbW9fc2luZ2xlX29iamVjdC5pcHluYiBleGFjdGx5OyB3ZSBvbmx5IHN3YXAgZmlsZSBJL08KZm9yIGluLW1lbW9yeSBieXRlcyBzbyB0aGUgcmVzdWx0IGNhbiB0cmF2ZWwgYmFjayB0aHJvdWdoIHRoZSBhc3luYyBlbmRwb2ludC4KCldlaWdodHM6IHRoZSBTMyBwcmVmaXggaXMgbW91bnRlZCBhdCAvb3B0L21sL21vZGVsLCBzbyBwaXBlbGluZS55YW1sICsKYWxsICouY2twdCBsaXZlIGZsYXQgdGhlcmUgKHNlZSBwcmVwYXJlX3dlaWdodHMucHkpLiBTQU0zRF9DT05GSUcgb3ZlcnJpZGVzLgoiIiIKCmltcG9ydCBiYXNlNjQKaW1wb3J0IGJpbmFzY2lpCmltcG9ydCBpbwppbXBvcnQgbG9nZ2luZwppbXBvcnQgb3MKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB0aW1lCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigic2FtM2QiKQpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyBbJShsZXZlbG5hbWUpc10gJShtZXNzYWdlKXMiKQoKIyBEZWZhdWx0IGxvY2F0aW9uIG9mIHRoZSBtb3VudGVkIHdlaWdodHMgcHJlZml4IChmbGF0dGVuZWQgY2hlY2twb2ludHMvIHRyZWUpLgpfQ09ORklHX1BBVEggPSBvcy5lbnZpcm9uLmdldCgiU0FNM0RfQ09ORklHIiwgIi9vcHQvbWwvbW9kZWwvcGlwZWxpbmUueWFtbCIpCgpfU1RBVEUgPSB7ImluZmVyZW5jZSI6IE5vbmUsICJoZWxwZXJzIjogTm9uZX0KCgpkZWYgX2xvYWRfaGVscGVycygpOgogICAgIiIiSW1wb3J0IHRoZSB1cHN0cmVhbSBwaXBlbGluZSBsYXppbHkgc28gaW1wb3J0IGVycm9ycyBzdXJmYWNlIGF0IGNvbGQgc3RhcnQKICAgIGluIENsb3VkV2F0Y2ggcmF0aGVyIHRoYW4gYXQgbW9kdWxlIGltcG9ydCB0aW1lIG9uIHRoZSBndW5pY29ybiBtYXN0ZXIuIiIiCiAgICAjIFRoZXNlIGNvbWUgZnJvbSAvb3B0L3NhbS0zZC1vYmplY3RzL25vdGVib29rIChvbiBQWVRIT05QQVRIKS4gU2FmZSBub3cgdGhhdAogICAgIyB0aGlzIGhhbmRsZXIgbW9kdWxlIGlzIE5PVCBuYW1lZCBgaW5mZXJlbmNlYC4KICAgIGZyb20gaW5mZXJlbmNlIGltcG9ydCAoICAjIG5vcWE6IEU0MDIKICAgICAgICBJbmZlcmVuY2UsCiAgICAgICAgbWFrZV9zY2VuZSwKICAgICAgICByZWFkeV9nYXVzc2lhbl9mb3JfdmlkZW9fcmVuZGVyaW5nLAogICAgICAgIHJlbmRlcl92aWRlbywKICAgICkKCiAgICByZXR1cm4gewogICAgICAgICJJbmZlcmVuY2UiOiBJbmZlcmVuY2UsCiAgICAgICAgIm1ha2Vfc2NlbmUiOiBtYWtlX3NjZW5lLAogICAgICAgICJyZWFkeV9nYXVzc2lhbl9mb3JfdmlkZW9fcmVuZGVyaW5nIjogcmVhZHlfZ2F1c3NpYW5fZm9yX3ZpZGVvX3JlbmRlcmluZywKICAgICAgICAicmVuZGVyX3ZpZGVvIjogcmVuZGVyX3ZpZGVvLAogICAgfQoKCmRlZiBsb2FkX21vZGVsKCk6CiAgICAiIiJCdWlsZCB0aGUgaW5mZXJlbmNlIHBpcGVsaW5lIG9uY2UgcGVyIHdvcmtlci4gSGVhdnkgKH4xMyBHQiBja3B0cyArIEdQVSkuIiIiCiAgICBpZiBfU1RBVEVbImluZmVyZW5jZSJdIGlzIG5vdCBOb25lOgogICAgICAgIHJldHVybiBfU1RBVEUKCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoX0NPTkZJR19QQVRIKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgZiJwaXBlbGluZSBjb25maWcgbm90IGZvdW5kIGF0IHtfQ09ORklHX1BBVEh9LiBEaWQgdGhlIFMzIHdlaWdodHMgcHJlZml4ICIKICAgICAgICAgICAgZiJtb3VudCBhdCAvb3B0L21sL21vZGVsPyBFeHBlY3RlZCBhIGZsYXR0ZW5lZCBjaGVja3BvaW50cy8gdHJlZSAiCiAgICAgICAgICAgIGYiKHBpcGVsaW5lLnlhbWwgKyAqLmNrcHQpLiBTZWUgcHJlcGFyZV93ZWlnaHRzLnB5LiIKICAgICAgICApCgogICAgaGVscGVycyA9IF9sb2FkX2hlbHBlcnMoKQogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgbG9nZ2VyLmluZm8oIkxvYWRpbmcgU0FNIDNEIE9iamVjdHMgcGlwZWxpbmUgZnJvbSAlcyAuLi4iLCBfQ09ORklHX1BBVEgpCiAgICBpbmZlcmVuY2UgPSBoZWxwZXJzWyJJbmZlcmVuY2UiXShfQ09ORklHX1BBVEgsIGNvbXBpbGU9RmFsc2UpCiAgICBsb2dnZXIuaW5mbygiUGlwZWxpbmUgbG9hZGVkICglLjFmcykiLCB0aW1lLnRpbWUoKSAtIHQwKQoKICAgIF9TVEFURVsiaW5mZXJlbmNlIl0gPSBpbmZlcmVuY2UKICAgIF9TVEFURVsiaGVscGVycyJdID0gaGVscGVycwogICAgcmV0dXJuIF9TVEFURQoKCmRlZiBfZGVjb2RlX2I2NF9pbWFnZShiNjQ6IHN0ciwgbW9kZTogc3RyKSAtPiBJbWFnZS5JbWFnZToKICAgIHRyeToKICAgICAgICByYXcgPSBiYXNlNjQuYjY0ZGVjb2RlKGI2NCwgdmFsaWRhdGU9VHJ1ZSkKICAgIGV4Y2VwdCAoYmluYXNjaWkuRXJyb3IsIFZhbHVlRXJyb3IpIGFzIGV4YzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBiYXNlNjQgcGF5bG9hZDoge2V4Y30iKSBmcm9tIGV4YwogICAgdHJ5OgogICAgICAgIGltZyA9IEltYWdlLm9wZW4oaW8uQnl0ZXNJTyhyYXcpKQogICAgICAgIGltZy5sb2FkKCkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJDb3VsZCBub3QgZGVjb2RlIGJ5dGVzIGFzIGFuIGltYWdlOiB7ZXhjfSIpIGZyb20gZXhjCiAgICByZXR1cm4gaW1nLmNvbnZlcnQobW9kZSkKCgpkZWYgcmVjb25zdHJ1Y3QoZGF0YTogZGljdCkgLT4gZGljdDoKICAgICIiIlJ1biB0aGUgZnVsbCBpbWFnZSttYXNrIC0+IDNEIHBpcGVsaW5lLiBgZGF0YWAgaXMgdGhlIHBhcnNlZCByZXF1ZXN0IEpTT04uIiIiCiAgICBzdGF0ZSA9IGxvYWRfbW9kZWwoKQogICAgaW5mZXJlbmNlID0gc3RhdGVbImluZmVyZW5jZSJdCiAgICBoZWxwZXJzID0gc3RhdGVbImhlbHBlcnMiXQoKICAgIGltYWdlX2I2NCA9IGRhdGEuZ2V0KCJpbWFnZSIpCiAgICBtYXNrX2I2NCA9IGRhdGEuZ2V0KCJtYXNrIikKICAgIGlmIG5vdCBpc2luc3RhbmNlKGltYWdlX2I2NCwgc3RyKSBvciBub3QgaW1hZ2VfYjY0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkZpZWxkICdpbWFnZScgKGJhc2U2NCBQTkcvSlBFRywgUkdCKSBpcyByZXF1aXJlZC4iKQogICAgaWYgbm90IGlzaW5zdGFuY2UobWFza19iNjQsIHN0cikgb3Igbm90IG1hc2tfYjY0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkZpZWxkICdtYXNrJyAoYmFzZTY0IHNpbmdsZS1jaGFubmVsIFBORywgMC8yNTUsIHNhbWUgSCpXIGFzIGltYWdlKSBpcyByZXF1aXJlZC4iKQoKICAgIHNlZWQgPSBpbnQoZGF0YS5nZXQoInNlZWQiLCA0MikpCiAgICByZW5kZXJfcHJldmlldyA9IGJvb2woZGF0YS5nZXQoInJlbmRlcl9wcmV2aWV3IiwgVHJ1ZSkpCiAgICBwcmV2aWV3X2ZyYW1lcyA9IGludChkYXRhLmdldCgicHJldmlld19mcmFtZXMiLCA0OCkpCiAgICBwcmV2aWV3X3Jlc29sdXRpb24gPSBpbnQoZGF0YS5nZXQoInByZXZpZXdfcmVzb2x1dGlvbiIsIDM4NCkpCgogICAgaW1hZ2UgPSBucC5hc2FycmF5KF9kZWNvZGVfYjY0X2ltYWdlKGltYWdlX2I2NCwgIlJHQiIpLCBkdHlwZT1ucC51aW50OCkKICAgIG1hc2tfaW1nID0gX2RlY29kZV9iNjRfaW1hZ2UobWFza19iNjQsICJMIikKICAgIGlmIG1hc2tfaW1nLnNpemUgIT0gKGltYWdlLnNoYXBlWzFdLCBpbWFnZS5zaGFwZVswXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgZiJtYXNrIHNpemUge21hc2tfaW1nLnNpemV9IChXeEgpIGRvZXMgbm90IG1hdGNoIGltYWdlICIKICAgICAgICAgICAgZiJ7KGltYWdlLnNoYXBlWzFdLCBpbWFnZS5zaGFwZVswXSl9LiBUaGV5IG11c3QgYWxpZ24gcGl4ZWwtZm9yLXBpeGVsLiIKICAgICAgICApCiAgICBtYXNrID0gbnAuYXNhcnJheShtYXNrX2ltZykgPiAwICAjIGJvb2xlYW4gSHhXLCBhcyB0aGUgcGlwZWxpbmUgZXhwZWN0cwoKICAgIHRpbWluZyA9IHt9CgogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgb3V0cHV0ID0gaW5mZXJlbmNlKGltYWdlLCBtYXNrLCBzZWVkPXNlZWQpCiAgICB0aW1pbmdbInJlY29uc3RydWN0Il0gPSByb3VuZCh0aW1lLnRpbWUoKSAtIHQwLCAyKQogICAgbG9nZ2VyLmluZm8oIlJlY29uc3RydWN0aW9uIGRvbmUgaW4gJS4yZnMiLCB0aW1pbmdbInJlY29uc3RydWN0Il0pCgogICAgIyBCdWlsZCBhIENFTlRFUkVELCB1bml0LW5vcm1hbGl6ZWQgc2NlbmUgZnJvbSB0aGUgb2JqZWN0IGJlZm9yZSBzYXZpbmcuIFJhdwogICAgIyBvYmplY3QtbG9jYWwgZ2F1c3NpYW5zIChvdXRwdXRbImdzIl0pIGFyZSBvZmYtY2VudGVyIGFuZCBhcmJpdHJhcmlseSBzY2FsZWQsCiAgICAjIHdoaWNoIG1ha2VzIHdlYiB2aWV3ZXJzIChnci5Nb2RlbDNEIGV0Yy4pIHNob3cgYmxhY2sgb3Igb25seSB0aWx0IHZlcnRpY2FsbHkuCiAgICAjIG1ha2Vfc2NlbmUgKyByZWFkeV9nYXVzc2lhbl9mb3JfdmlkZW9fcmVuZGVyaW5nIGNlbnRlcnMgKyBub3JtYWxpemVzIGl0LCBzbwogICAgIyBhbnkgb3JiaXQtY2FtZXJhIHZpZXdlciBmcmFtZXMgaXQgYW5kIHJvdGF0ZXMgbGVmdC9yaWdodCBjb3JyZWN0bHkuIFRoaXMgaXMKICAgICMgZXhhY3RseSB0aGUgcmVwbydzIGRlbW9fc2luZ2xlX29iamVjdC5pcHluYiB2aXN1YWxpemF0aW9uIHBhdGguCiAgICBzY2VuZV9ncyA9IGhlbHBlcnNbIm1ha2Vfc2NlbmUiXShvdXRwdXQpCiAgICBzY2VuZV9ncyA9IGhlbHBlcnNbInJlYWR5X2dhdXNzaWFuX2Zvcl92aWRlb19yZW5kZXJpbmciXShzY2VuZV9ncykKCiAgICB3aXRoIHRlbXBmaWxlLk5hbWVkVGVtcG9yYXJ5RmlsZShzdWZmaXg9Ii5wbHkiLCBkZWxldGU9RmFsc2UpIGFzIHRmOgogICAgICAgIHBseV9wYXRoID0gdGYubmFtZQogICAgdHJ5OgogICAgICAgIHNjZW5lX2dzLnNhdmVfcGx5KHBseV9wYXRoKQogICAgICAgIHdpdGggb3BlbihwbHlfcGF0aCwgInJiIikgYXMgZmg6CiAgICAgICAgICAgIHBseV9ieXRlcyA9IGZoLnJlYWQoKQogICAgZmluYWxseToKICAgICAgICB0cnk6CiAgICAgICAgICAgIG9zLnJlbW92ZShwbHlfcGF0aCkKICAgICAgICBleGNlcHQgT1NFcnJvcjoKICAgICAgICAgICAgcGFzcwoKICAgIG51bV9nYXVzc2lhbnMgPSBOb25lCiAgICB0cnk6CiAgICAgICAgbnVtX2dhdXNzaWFucyA9IGludChzY2VuZV9ncy5nZXRfeHl6LnNoYXBlWzBdKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwYXNzCgogICAgcHJldmlld19naWZfYjY0ID0gTm9uZQogICAgaWYgcmVuZGVyX3ByZXZpZXc6CiAgICAgICAgdHJ5OgogICAgICAgICAgICB0MSA9IHRpbWUudGltZSgpCiAgICAgICAgICAgIHZpZGVvID0gaGVscGVyc1sicmVuZGVyX3ZpZGVvIl0oCiAgICAgICAgICAgICAgICBzY2VuZV9ncywKICAgICAgICAgICAgICAgIHI9MSwKICAgICAgICAgICAgICAgIGZvdj02MCwKICAgICAgICAgICAgICAgIHBpdGNoX2RlZz0xNSwKICAgICAgICAgICAgICAgIHlhd19zdGFydF9kZWc9LTQ1LAogICAgICAgICAgICAgICAgcmVzb2x1dGlvbj1wcmV2aWV3X3Jlc29sdXRpb24sCiAgICAgICAgICAgICAgICBudW1fZnJhbWVzPXByZXZpZXdfZnJhbWVzLAogICAgICAgICAgICApWyJjb2xvciJdCiAgICAgICAgICAgIGltcG9ydCBpbWFnZWlvCiAgICAgICAgICAgIGJ1ZiA9IGlvLkJ5dGVzSU8oKQogICAgICAgICAgICBpbWFnZWlvLm1pbXNhdmUoYnVmLCB2aWRlbywgZm9ybWF0PSJHSUYiLCBkdXJhdGlvbj0xMDAwIC8gMzAsIGxvb3A9MCkKICAgICAgICAgICAgcHJldmlld19naWZfYjY0ID0gYmFzZTY0LmI2NGVuY29kZShidWYuZ2V0dmFsdWUoKSkuZGVjb2RlKCJhc2NpaSIpCiAgICAgICAgICAgIHRpbWluZ1sicmVuZGVyIl0gPSByb3VuZCh0aW1lLnRpbWUoKSAtIHQxLCAyKQogICAgICAgICAgICBsb2dnZXIuaW5mbygiVHVybnRhYmxlIHJlbmRlciBkb25lIGluICUuMmZzIiwgdGltaW5nWyJyZW5kZXIiXSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzogICMgcHJldmlldyBpcyBiZXN0LWVmZm9ydDsgbmV2ZXIgZmFpbCB0aGUgM0QgcmVzdWx0IG9uIGl0CiAgICAgICAgICAgIGxvZ2dlci53YXJuaW5nKCJQcmV2aWV3IHJlbmRlciBmYWlsZWQgKHJldHVybmluZyAucGx5IG9ubHkpOiAlcyIsIGV4YykKCiAgICByZXR1cm4gewogICAgICAgICJwbHlfYjY0IjogYmFzZTY0LmI2NGVuY29kZShwbHlfYnl0ZXMpLmRlY29kZSgiYXNjaWkiKSwKICAgICAgICAibnVtX2dhdXNzaWFucyI6IG51bV9nYXVzc2lhbnMsCiAgICAgICAgInByZXZpZXdfZ2lmX2I2NCI6IHByZXZpZXdfZ2lmX2I2NCwKICAgICAgICAidGltaW5nX3MiOiB0aW1pbmcsCiAgICB9Cg==", "serve/serve": "IyEvYmluL2Jhc2gKIyBTYWdlTWFrZXIgaG9zdGluZyBlbnRyeXBvaW50LiBTYWdlTWFrZXIgcnVucyBgZG9ja2VyIHJ1biA8aW1hZ2U+IHNlcnZlYDsgdGhpcwojIHNjcmlwdCBhY3RpdmF0ZXMgdGhlIGNvbmRhIGVudiB0aGF0IGhvbGRzIHRoZSBjb21waWxlZCBTQU0gM0QgT2JqZWN0cyBzdGFjayBhbmQKIyBsYXVuY2hlcyB0aGUgRmxhc2sgYXBwIHVuZGVyIGd1bmljb3JuIG9uIHRoZSByZXF1aXJlZCBwb3J0IDgwODAuCiMKIyBTaW5nbGUgd29ya2VyOiB0aGUgbW9kZWwgaXMgbGFyZ2UgYW5kIHNpbmdsZS1HUFUsIHNvIG9uZSByZXNpZGVudCBwcm9jZXNzIG93bnMKIyB0aGUgR1BVLiB0aW1lb3V0PTAgYmVjYXVzZSBhIGNvbGQgbW9kZWwgbG9hZCArIDNEIHJlY29uc3RydWN0aW9uICsgdHVybnRhYmxlCiMgcmVuZGVyIGNhbiB0YWtlIG1pbnV0ZXMgKHRoZSBTYWdlTWFrZXIgYXN5bmMgaW52b2NhdGlvbiBoYXMgaXRzIG93biB0aW1lb3V0KS4Kc2V0IC1lCgojIE1ha2UgYG1hbWJhIHJ1bmAvY29uZGEgYWN0aXZhdGlvbiBhdmFpbGFibGUsIHRoZW4gYWN0aXZhdGUgdGhlIGVudi4Kc291cmNlIC9vcHQvY29uZGEvZXRjL3Byb2ZpbGUuZC9jb25kYS5zaApjb25kYSBhY3RpdmF0ZSBzYW0zZC1vYmplY3RzCgpNT0RFTF9TRVJWRVJfVElNRU9VVD0ke01PREVMX1NFUlZFUl9USU1FT1VUOi0wfQpNT0RFTF9TRVJWRVJfV09SS0VSUz0ke01PREVMX1NFUlZFUl9XT1JLRVJTOi0xfQoKZXhlYyBndW5pY29ybiBcCiAgICAtLWJpbmQgMC4wLjAuMDo4MDgwIFwKICAgIC0td29ya2VycyAiJHtNT0RFTF9TRVJWRVJfV09SS0VSU30iIFwKICAgIC0tdGhyZWFkcyAxIFwKICAgIC0tdGltZW91dCAiJHtNT0RFTF9TRVJWRVJfVElNRU9VVH0iIFwKICAgIC0tZ3JhY2VmdWwtdGltZW91dCA2MCBcCiAgICAtLWtlZXAtYWxpdmUgNjUgXAogICAgLS1hY2Nlc3MtbG9nZmlsZSAtIFwKICAgIC0tZXJyb3ItbG9nZmlsZSAtIFwKICAgIHByZWRpY3RvcjphcHAK"}""")
for rel, b64 in _FILES.items():
    path = os.path.join('container', rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'wb') as f:
        f.write(base64.b64decode(b64))

# Remove the stale, colliding handler + bytecode if present.
for stale in ['container/serve/inference.py', 'container/serve/__pycache__']:
    if os.path.isdir(stale):
        shutil.rmtree(stale)
    elif os.path.exists(stale):
        os.remove(stale)
os.chmod('container/serve/serve', 0o755)

print('Wrote container files:', sorted(_FILES))
print('container/serve now contains:', sorted(os.listdir('container/serve')))
assert not os.path.exists('container/serve/inference.py'), 'stale inference.py still present!'
print('OK: build context is correct (sam3d_handler.py present, inference.py absent).')


In [ ]:
# Robust build: call the CLI in-process (no reliance on `sm-docker` being on the
# shell PATH, which it often isn't in notebook `!` subshells). Auto-installs the
# package into THIS kernel if needed. Streams CodeBuild logs until the build ends.
import os, sys, subprocess
try:
    from sagemaker_studio_image_build import cli
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sagemaker-studio-image-build'])
    from sagemaker_studio_image_build import cli

argv = ['sm-docker', 'build', '.',
        '--repository', f'{ECR_REPO}:latest',
        '--compute-type', 'BUILD_GENERAL1_2XLARGE',
        '--build-arg', 'SAM3D_COMMIT=main']   # pin a commit instead of 'main' for reproducibility

_old_argv, _cwd = sys.argv, os.getcwd()
os.chdir('container')                          # sm-docker packages the CWD as the build context
try:
    sys.argv = argv
    cli.main()
    print('\nBuild submitted/finished. The image URI matches IMAGE_URI above.')
except SystemExit as e:
    print(f'\nsm-docker exited with code {e.code}.' if e.code else '\nBuild finished.')
finally:
    sys.argv, _ = _old_argv, os.chdir(_cwd)

# Tip: this cell blocks while CodeBuild runs. To watch from a terminal instead,
# run:  python monitor.py build --follow
# If flash_attn compile exceeds the CodeBuild timeout, raise the timeout on the
# auto-created 'sagemaker-studio-*' CodeBuild project and re-run.

In [ ]:
# Confirm the image is in ECR before deploying (boto3 avoids the brace-collision
# that breaks `!aws ... --query '{...}'` inside notebook shell cells).
ecr = boto3.client('ecr', region_name=region)
try:
    imgs = ecr.describe_images(repositoryName=ECR_REPO)['imageDetails']
    latest = sorted(imgs, key=lambda i: i['imagePushedAt'])[-1]
    print('Image present in ECR repo:', ECR_REPO)
    print('  tags  :', latest.get('imageTags'))
    print('  pushed:', latest['imagePushedAt'])
    print('  size  : %.1f MB' % (latest['imageSizeInBytes'] / 1e6))
    print('  digest:', latest['imageDigest'])
    print('\nReady to deploy (section 6). IMAGE_URI =', IMAGE_URI)
except ecr.exceptions.RepositoryNotFoundException:
    print(f'Repository {ECR_REPO} not found — did section 5 finish successfully?')
except Exception as e:
    print('Could not describe images:', e)

## 6. Deploy the async endpoint
BYOC `Model` + uncompressed-prefix `model_data` (mounts the checkpoints at `/opt/ml/model`). Async because a base64 image + the returned `.ply`/GIF exceed real-time limits. Generous timeouts cover the heavy cold load.

> Cold start is ~minutes (13 GB weights + GPU load). **Watch it from a terminal:** `python monitor.py endpoint --follow` — it prints status, `FailureReason`, and CloudWatch logs, and stops on its own at `InService`/`Failed`.

In [ ]:
from sagemaker.model import Model
from sagemaker.async_inference import AsyncInferenceConfig

# Delete any existing endpoint/config/model with this name so a re-run picks up
# the freshly built image. Idempotent — ignores 'not found'.
sm = boto3.client('sagemaker', region_name=region)
for _del, _kw in [(sm.delete_endpoint, {'EndpointName': ENDPOINT_NAME}),
                  (sm.delete_endpoint_config, {'EndpointConfigName': ENDPOINT_NAME}),
                  (sm.delete_model, {'ModelName': ENDPOINT_NAME})]:
    try:
        _del(**_kw)
        print('deleted', list(_kw.values())[0])
    except Exception:
        pass
import time as _t; _t.sleep(10)  # let the deletes settle before recreating

model = Model(
    image_uri=IMAGE_URI,
    model_data={'S3DataSource': {
        'S3Uri': WEIGHTS_PREFIX, 'S3DataType': 'S3Prefix', 'CompressionType': 'None'}},
    role=role,
    env={'SAM3D_CONFIG': '/opt/ml/model/pipeline.yaml'},
    sagemaker_session=sess,
    name=ENDPOINT_NAME,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
    async_inference_config=AsyncInferenceConfig(
        output_path=OUTPUT_PREFIX, failure_path=FAILURE_PREFIX,
        max_concurrent_invocations_per_instance=1),
    container_startup_health_check_timeout=3600,
    model_data_download_timeout=3600,
)
print('Deployed:', ENDPOINT_NAME)

In [ ]:
# Watch status. 'Creating' = still working; 'InService' = ready; 'Failed' = read FailureReason.
sm = boto3.client('sagemaker', region_name=region)
d = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
print(d['EndpointStatus'], '|', d.get('FailureReason', ''))

## 7. Async invoke helper
Uploads the request JSON to S3, calls `invoke_endpoint_async`, polls the output/failure locations.

In [ ]:
import time, uuid
from urllib.parse import urlparse

s3 = boto3.client('s3', region_name=region)
smr = boto3.client('sagemaker-runtime', region_name=region)

def _split(uri):
    p = urlparse(uri); return p.netloc, p.path.lstrip('/')

def invoke_async(payload: dict, timeout_s: int = 1800, poll_s: float = 5.0) -> dict:
    in_bucket, in_prefix = _split(INPUT_PREFIX)
    key = f'{in_prefix}{uuid.uuid4().hex}.json'
    s3.put_object(Bucket=in_bucket, Key=key, Body=json.dumps(payload).encode(),
                  ContentType='application/json')
    resp = smr.invoke_endpoint_async(
        EndpointName=ENDPOINT_NAME, InputLocation=f's3://{in_bucket}/{key}',
        ContentType='application/json', InvocationTimeoutSeconds=3600)
    ob, ok = _split(resp['OutputLocation'])
    fb, fk = _split(resp['FailureLocation']) if resp.get('FailureLocation') else (None, None)
    print('submitted; polling', resp['OutputLocation'])
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            return json.loads(s3.get_object(Bucket=ob, Key=ok)['Body'].read())
        except s3.exceptions.NoSuchKey:
            pass
        if fb:
            try:
                raise RuntimeError('endpoint failure: ' + s3.get_object(Bucket=fb, Key=fk)['Body'].read().decode())
            except s3.exceptions.NoSuchKey:
                pass
        time.sleep(poll_s)
    raise TimeoutError(f'no result after {timeout_s}s')

## 8. Mark the object (click → mask)
We run a small local **`facebook/sam-vit-base`**: give it a point on the object, it returns a mask. We display the overlay, then send `image` + `mask` to the 3D endpoint. Uses the repo's sample photo so it runs without your own image.

In [ ]:
import urllib.request, numpy as np
from PIL import Image

SAMPLE_URL = ('https://raw.githubusercontent.com/facebookresearch/sam-3d-objects/main/'
              'notebook/images/shutterstock_stylish_kidsroom_1640806567/image.png')
IMAGE_PATH = 'sample.png'
urllib.request.urlretrieve(SAMPLE_URL, IMAGE_PATH)
image = Image.open(IMAGE_PATH).convert('RGB')
print('image size (WxH):', image.size)
image

In [ ]:
POINT_XY = (470, 300)   # <-- a point ON the object you want in 3D (read coords off the image above)

import torch
from transformers import SamModel, SamProcessor

_dev = 'cuda' if torch.cuda.is_available() else 'cpu'
_sam = SamModel.from_pretrained('facebook/sam-vit-base').to(_dev).eval()
_proc = SamProcessor.from_pretrained('facebook/sam-vit-base')

def mask_from_point(img, xy):
    inp = _proc(img, input_points=[[[float(xy[0]), float(xy[1])]]], return_tensors='pt').to(_dev)
    with torch.no_grad():
        out = _sam(**inp)
    masks = _proc.image_processor.post_process_masks(
        out.pred_masks.cpu(), inp['original_sizes'].cpu(), inp['reshaped_input_sizes'].cpu())[0][0]
    return masks[int(out.iou_scores[0, 0].argmax())].numpy().astype(bool)

mask = mask_from_point(image, POINT_XY)
print('mask covers', int(mask.sum()), 'px')

In [ ]:
import matplotlib.pyplot as plt
ov = np.array(image).copy()
ov[mask] = (0.5 * ov[mask] + 0.5 * np.array([0, 200, 255])).astype(np.uint8)
plt.figure(figsize=(7, 7)); plt.imshow(ov)
plt.scatter([POINT_XY[0]], [POINT_XY[1]], c='red', s=120, marker='*', edgecolors='white')
plt.title(f'object @ {POINT_XY}'); plt.axis('off'); plt.show()
# Wrong object? Change POINT_XY and re-run the previous cell + this one.

## 9. Reconstruct 3D + preview
Send image + mask to the endpoint; get the Gaussian-splat `.ply` and a server-rendered turntable GIF.

> If a call errors or hangs, inspect server-side failures from a terminal: `python monitor.py failures`

In [ ]:
import base64, io

def b64_png(arr_or_img):
    img = arr_or_img if isinstance(arr_or_img, Image.Image) else Image.fromarray(arr_or_img)
    buf = io.BytesIO(); img.save(buf, format='PNG'); return base64.b64encode(buf.getvalue()).decode()

payload = {
    'image': b64_png(image),
    'mask':  b64_png((mask.astype('uint8') * 255)),
    'seed': 42, 'render_preview': True, 'preview_frames': 48, 'preview_resolution': 384,
}
result = invoke_async(payload, timeout_s=1800)
print('gaussians:', result.get('num_gaussians'), '| timing_s:', result.get('timing_s'))

In [ ]:
from IPython.display import Image as IPyImage, display

ply_bytes = base64.b64decode(result['ply_b64'])
open('reconstruction.ply', 'wb').write(ply_bytes)
print('wrote reconstruction.ply (%.1f MB)' % (len(ply_bytes) / 1e6))

if result.get('preview_gif_b64'):
    gif = base64.b64decode(result['preview_gif_b64'])
    open('reconstruction.gif', 'wb').write(gif)
    display(IPyImage(data=gif, format='gif'))
else:
    print('No preview returned (render off or rendering failed).')

## 10. View / use the 3D model
`reconstruction.ply` is a 3D Gaussian Splat. View it in any 3DGS viewer:
- **[PlayCanvas SuperSplat](https://superspl.at/editor)** — drag the `.ply` in (browser).
- **[antimatter15/splat](https://antimatter15.com/splat/)** — drag-and-drop viewer.

The inline GIF is the quick preview; the `.ply` is the real, manipulable asset.

## 11. Interactive web app (click-to-mask + rotatable 3D)
A Gradio web UI that reproduces the Meta demo: **click** the object (click again to refine — switch to *Remove* to subtract), press **Generate 3D**, then **drag to rotate** / scroll to zoom the result. Reuses this notebook's `invoke_async` and deployed endpoint.

Requires sections 1 (config) and 7 (invoke helper) to have run, and the endpoint `InService`.

In [ ]:
%pip install -q 'gradio>=4.31'
import importlib, app as sam3d_app
importlib.reload(sam3d_app)
# Reuse the notebook's invoke_async (section 7) so it targets THIS endpoint/config.
demo = sam3d_app.build_demo(invoke_fn=invoke_async)
# share=True gives a public gradio.live link (easiest in SageMaker). If your
# environment blocks it, drop share=True and open the printed local URL via the
# Jupyter proxy, or run `python app.py` from a terminal instead.
demo.launch(share=True)

## 12. Cleanup (important — the GPU endpoint is the main cost)
Delete the endpoint when idle. Keep the `sam3d/weights/` prefix for cheap redeploys.

In [ ]:
# sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
# sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
# sm.delete_model(ModelName=ENDPOINT_NAME)
# print('deleted', ENDPOINT_NAME)